# E20260914081253343514: E060 epoch sweep

Этот template копируется командой `make new-experiment`. Вставьте код обучения в одну функцию и нажмите **Run All** — остальное обрабатывается автоматически.

In [ ]:
# Setup: найдите корень клонированного репозитория и импортируйте runner.
from __future__ import annotations

import importlib.util
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "configs/project.json").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Откройте notebook внутри клонированного репозитория.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
if importlib.util.find_spec("clearml") is None:
    print("WARNING: установите ClearML один раз: pip install -e '.[tracking]'")
from pmldl_llm import (
    ExperimentOutput,
    load_experiment_setup,
    run_notebook_experiment,
)

PROJECT_ROOT

## 1. Готовый конфиг эксперимента

ID, owner, parent, seed, split и ClearML уже настроены командой `make new-experiment`. Эту ячейку менять не нужно.

In [ ]:
SETUP = load_experiment_setup("configs/experiments/E20260914081253343514.json", project_root=PROJECT_ROOT)
SETUP

## 2. Ваш эксперимент

Перенесите существующий код обучения и оценки внутрь функции. В конце верните итоговые validation-метрики и пути к файлам, которые нужно сохранить.

In [ ]:
# ============================================================================
# Cell 2 тренировочного notebook (make new-experiment) — E060, sweep эпох.
#
# Логика повторяет scripts/train_symmetric_encoder.py 1-в-1 (тот же
# backbone/split/batch size/truncation/архитектура из configs/symmetric_encoder.json),
# но:
#   - обучение идёт максимум до 4 эпох;
#   - после 2, 3 и 4 эпохи (если запущена) считается validation-качество
#     и логируется в history через run.log_metric(..., step=epoch);
#   - финально выбирается эпоха с лучшим (минимальным) log loss, и именно
#     её probabilities/checkpoint становятся официальным результатом run'а.
#
# Единственное отличие от общего конфига: seed зафиксирован в 20260903
# (тот же seed, что у всех E000-E011 и у исходного E060) — поставь его
# в configs/experiments/<ID>.json после `make new-experiment`, до запуска.
# ============================================================================

import hashlib
import json

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

from pmldl_llm.config import validate_fold_roles
from pmldl_llm.constants import (
    DEFAULT_CHECKSUM_MANIFEST,
    DEFAULT_DATA_DIR,
    DEFAULT_FOLD_PATH,
    DEFAULT_SPLIT_CONFIG,
    TARGET_COLUMNS,
)
from pmldl_llm.data import (
    load_competition_data,
    swap_probability_columns,
    swap_target_indices,
    swapped_frame,
    target_indices,
    verify_competition_data_dir,
    verify_checksum_manifest,
)
from pmldl_llm.evaluation import (
    evaluate_experiment_probabilities,
    normalize_probabilities,
)
from pmldl_llm.split import load_frozen_folds
from pmldl_llm.symmetric_model import SymmetricPreferenceModel
from pmldl_llm.text import flatten_conversation
from pmldl_llm.truncation import balanced_head_tail_truncate

# Единственные "тюнимые" настройки этого эксперимента: сколько эпох пробуем.
EPOCH_CHECKPOINTS = [2, 3, 4]

# Всё остальное = configs/symmetric_encoder.json из E060, без изменений.
MODEL_CONFIG = {
    "model_name": "microsoft/deberta-v3-small",
    "max_length": 256,
    "special_tokens": 3,
    "prompt_weight": 1,
    "response_weight": 2,
    "batch_size": 16,
    "eval_batch_size": 32,
    "learning_rate": 2e-5,
    "head_learning_rate": 1e-3,
    "warmup_ratio": 0.06,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
}


def encode_side(tokenizer, prompt_text, response_text, cfg):
    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    response_ids = tokenizer.encode(response_text, add_special_tokens=False)
    truncated_prompt, truncated_response, _empty, _stats = balanced_head_tail_truncate(
        prompt_ids,
        response_ids,
        [],
        max_length=cfg["max_length"],
        special_tokens=cfg["special_tokens"],
        budget_weights=(cfg["prompt_weight"], cfg["response_weight"], cfg["response_weight"]),
    )
    cls_id, sep_id = tokenizer.cls_token_id, tokenizer.sep_token_id
    input_ids = [cls_id] + truncated_prompt + [sep_id] + truncated_response + [sep_id]
    attention_mask = [1] * len(input_ids)
    pad_length = cfg["max_length"] - len(input_ids)
    if pad_length > 0:
        input_ids += [tokenizer.pad_token_id] * pad_length
        attention_mask += [0] * pad_length
    return input_ids[: cfg["max_length"]], attention_mask[: cfg["max_length"]]


class PreferencePairDataset(Dataset):
    def __init__(self, frame, tokenizer, targets, cfg):
        self.prompts = [flatten_conversation(v) for v in frame["prompt"]]
        self.responses_a = [flatten_conversation(v) for v in frame["response_a"]]
        self.responses_b = [flatten_conversation(v) for v in frame["response_b"]]
        self.targets = targets
        self.tokenizer = tokenizer
        self.cfg = cfg

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, index):
        ids_a, mask_a = encode_side(self.tokenizer, self.prompts[index], self.responses_a[index], self.cfg)
        ids_b, mask_b = encode_side(self.tokenizer, self.prompts[index], self.responses_b[index], self.cfg)
        item = {
            "input_ids_a": torch.tensor(ids_a, dtype=torch.long),
            "attention_mask_a": torch.tensor(mask_a, dtype=torch.long),
            "input_ids_b": torch.tensor(ids_b, dtype=torch.long),
            "attention_mask_b": torch.tensor(mask_b, dtype=torch.long),
        }
        if self.targets is not None:
            item["target"] = torch.tensor(self.targets[index], dtype=torch.long)
        return item


@torch.no_grad()
def predict_probabilities(model, frame, tokenizer, cfg, device):
    model.eval()

    def run_forward(data_frame):
        dataset = PreferencePairDataset(data_frame, tokenizer, None, cfg)
        loader = DataLoader(dataset, batch_size=cfg["eval_batch_size"])
        probs = []
        for batch in loader:
            logits = model(
                batch["input_ids_a"].to(device), batch["attention_mask_a"].to(device),
                batch["input_ids_b"].to(device), batch["attention_mask_b"].to(device),
            )
            probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
        return normalize_probabilities(np.concatenate(probs, axis=0))

    original = run_forward(frame)
    swapped_back = swap_probability_columns(run_forward(swapped_frame(frame)))
    return original, swapped_back


def train_one_epoch(model, loader, optimizer, scheduler, device, max_grad_norm):
    model.train()
    loss_fn = torch.nn.CrossEntropyLoss()
    total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        logits = model(
            batch["input_ids_a"].to(device), batch["attention_mask_a"].to(device),
            batch["input_ids_b"].to(device), batch["attention_mask_b"].to(device),
        )
        loss = loss_fn(logits, batch["target"].to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        scheduler.step()
        total_loss += float(loss.item())
    return total_loss / max(len(loader), 1)


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def train_and_evaluate(run):
    cfg = MODEL_CONFIG
    is_smoke = bool(run.config.get("smoke_test"))
    # Smoke test = fast sanity check of the wiring, not a real training run:
    # tiny data slice, a single epoch. The real sweep only happens once
    # `make prepare-full` flips smoke_test to false and you Run All again.
    epoch_checkpoints = [1] if is_smoke else EPOCH_CHECKPOINTS
    max_epochs = max(epoch_checkpoints)
    smoke_rows = 32

    split_config = json.loads(DEFAULT_SPLIT_CONFIG.read_text(encoding="utf-8"))
    roles = validate_fold_roles(split_config)
    dataset_hashes = verify_checksum_manifest(DEFAULT_CHECKSUM_MANIFEST)
    verify_competition_data_dir(DEFAULT_DATA_DIR, dataset_hashes)
    train, _test = load_competition_data(DEFAULT_DATA_DIR)
    folds = load_frozen_folds(
        train, DEFAULT_FOLD_PATH, DEFAULT_SPLIT_CONFIG,
        n_splits=roles.n_splits, dataset_hashes=dataset_hashes,
    )
    fold_values = folds["fold"].to_numpy()
    is_training = np.isin(fold_values, list(roles.training_folds))
    is_validation = fold_values == roles.validation_fold

    y = target_indices(train)
    training_frame = train.loc[is_training].reset_index(drop=True)
    validation_frame = train.loc[is_validation].reset_index(drop=True)
    y_train = y[is_training]
    y_validation = y[is_validation]

    if is_smoke:
        training_frame = training_frame.iloc[:smoke_rows].reset_index(drop=True)
        y_train = y_train[:smoke_rows]
        validation_frame = validation_frame.iloc[:smoke_rows].reset_index(drop=True)
        y_validation = y_validation[:smoke_rows]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"])
    model = SymmetricPreferenceModel(cfg["model_name"]).to(device)

    augmented_frame = pd.concat([training_frame, swapped_frame(training_frame)], ignore_index=True)
    augmented_targets = np.concatenate([y_train, swap_target_indices(y_train)])
    train_loader = DataLoader(
        PreferencePairDataset(augmented_frame, tokenizer, augmented_targets, cfg),
        batch_size=cfg["batch_size"], shuffle=True,
    )

    optimizer = torch.optim.AdamW(
        [
            {"params": model.encoder.parameters(), "lr": cfg["learning_rate"]},
            {"params": list(model.preference_head.parameters()) + list(model.tie_head.parameters()),
             "lr": cfg["head_learning_rate"]},
        ],
        weight_decay=cfg["weight_decay"],
    )
    total_steps = len(train_loader) * max_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * cfg["warmup_ratio"]),
        num_training_steps=total_steps,
    )

    best_epoch = None
    best_log_loss = float("inf")
    best_original = best_swapped_back = None
    per_epoch_results = {}

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, device, cfg["max_grad_norm"])
        run.log_metric("train_loss", train_loss, namespace="train", step=epoch)

        if epoch in epoch_checkpoints:
            original, swapped_back = predict_probabilities(model, validation_frame, tokenizer, cfg, device)
            epoch_metrics = evaluate_experiment_probabilities(y_validation, original, swapped_back)
            run.log_metrics(epoch_metrics, namespace="validation", step=epoch)
            per_epoch_results[epoch] = epoch_metrics
            print(f"epoch {epoch}: {epoch_metrics}")

            if epoch_metrics["log_loss"] < best_log_loss:
                best_log_loss = epoch_metrics["log_loss"]
                best_epoch = epoch
                best_original, best_swapped_back = original, swapped_back
                # Keep only the best checkpoint on disk.
                torch.save(model.state_dict(), run.artifact_path("checkpoint/model_state_dict.pt"))
            else:
                print(
                    f"epoch {epoch} did not improve log loss "
                    f"({epoch_metrics['log_loss']:.6f} >= {best_log_loss:.6f}); "
                    "keeping the previous best checkpoint."
                )
                if epoch < max_epochs and epoch != epoch_checkpoints[-1]:
                    # Optional early stop: comment this out to always go to max_epochs.
                    break

    fold7_path = run.artifact_path("fold7_predictions.csv")
    pd.DataFrame({
        "id": validation_frame["id"].to_numpy(),
        "target": y_validation,
        TARGET_COLUMNS[0]: best_original[:, 0],
        TARGET_COLUMNS[1]: best_original[:, 1],
        TARGET_COLUMNS[2]: best_original[:, 2],
    }).to_csv(fold7_path, index=False)

    manifest_path = run.artifact_path("model_manifest.json")
    manifest_path.write_text(json.dumps({
        "run_id": run.run_id,
        "experiment_id": run.config["experiment_id"],
        "seed": run.config["seed"],
        "best_epoch": best_epoch,
        "epochs_tried": sorted(per_epoch_results),
        "class_order": list(TARGET_COLUMNS),
        "checkpoint_path": "checkpoint/model_state_dict.pt",
        "checkpoint_sha256": sha256_of(run.artifact_dir / "checkpoint" / "model_state_dict.pt"),
    }, indent=2) + "\n", encoding="utf-8")

    print(f"Best epoch: {best_epoch} (log_loss={best_log_loss:.6f})")
    print(f"Per-epoch results: {json.dumps(per_epoch_results, indent=2)}")

    return ExperimentOutput.from_predictions(
        y_true=y_validation,
        original_probabilities=best_original,
        swapped_back_probabilities=best_swapped_back,
        artifacts={
            # Already written in place above via run.artifact_path(); listing
            # them here just makes sure the ClearML tracker uploads each one
            # too (log_artifact skips the redundant local copy when source
            # and destination already match).
            "checkpoint/model_state_dict.pt": run.artifact_dir / "checkpoint" / "model_state_dict.pt",
            "model_manifest.json": manifest_path,
            "fold7_predictions.csv": fold7_path,
        },
    )


## 3. Автоматический запуск

Эту ячейку менять не нужно. При ошибке run автоматически сохранится со статусом `failed`; при успехе он будет проверен и попадёт в ClearML, а full-run — в локальный leaderboard.

In [ ]:
RESULT = run_notebook_experiment(
    train_and_evaluate,
    SETUP,
    project_root=PROJECT_ROOT,
)
RESULT

## Что получится

- `configs/experiments/<experiment_id>.json` — созданный конфиг;
- `results/runs/<run_id>/` — проверенные метрики и metadata;
- `artifacts/<run_id>/` — модели и predictions;
- ClearML task — live-метрики;
- `results/leaderboard.csv` — автоматически обновлённое локальное сравнение.

Первый **Run All** — smoke-test. Затем выполните `make prepare-full EXPERIMENT=<ID>`, перезапустите kernel и снова нажмите **Run All**. После успеха выполните `make submit-experiment EXPERIMENT=<ID>`.